```
ACTUAL FINAL PREDICTION WILL BE DONE HERE

- IN THIS FILE WE WILL IMPORT THE PRETRAINED WEIGHTS FOR DATA WITH +-19.1 ERROR.

- THEN WE WILL TEST IT AGAINST THE NEW DATA FROM RESHMA.
< https://drive.google.com/file/d/1GglAoC7cyk_yIBkPqAuPZR__aGVO4wwx/view?usp=sharing >

```


In [ ]:
#1
import torch
import torch.nn as nn
from torchvision import models

In [ ]:
#2
class CNNEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        base = models.resnet18(weights=None) # Set weights=None as we are loading custom weights
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, out_dim)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

class CNN_LSTM_AQI(nn.Module):
    def __init__(self, img_feat_dim=128):
        super().__init__()

        self.cnn = CNNEncoder(out_dim=img_feat_dim)
        self.lstm = nn.LSTM(
            input_size=img_feat_dim + 1,
            hidden_size=64,
            batch_first=True
        )
        self.fc = nn.Linear(64, 1)

    def forward(self, img_seq, aqi_seq):
        B, T, C, H, W = img_seq.shape
        img_seq_flat = img_seq.view(B*T, C, H, W)

        img_feats_flat = self.cnn(img_seq_flat)
        img_feats = img_feats_flat.view(B, T, -1)

        x = torch.cat([img_feats, aqi_seq], dim=-1)
        out, _ = self.lstm(x)

        return self.fc(out[:, -1])

In [ ]:
 #3
 #CNN ENCODER CALLED - PREVIOUSLY TRAINED

model_save_path = '/content/drive/MyDrive/Amajor/img_summ/7_days_imgs_inp/aqi_new_model.pth'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN_LSTM_AQI(img_feat_dim=128).to(device)

# Load the state dictionary
model.load_state_dict(torch.load(model_save_path, map_location=device))
model.eval()

print(f"Model weights loaded successfully from {model_save_path} to {device}.")

Model weights loaded successfully from /content/drive/MyDrive/Amajor/img_summ/7_days_imgs_inp/aqi_new_model.pth to cuda.


from here iqra & reshma will have to add some parts.

In [ ]:
#s1
import pandas as pd
import numpy as np
import torch
from PIL import Image
import torchvision.transforms as T
import os

In [ ]:
#s2
df = pd.read_csv("/content/drive/MyDrive/Amajor/img_summ/check_model/data_check.csv")

validation_df = df.copy()
validation_df["predicted_aqi"] = np.nan

In [ ]:
#s3
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

def load_image(path):
    if pd.isna(path) or not os.path.exists(path):
        return torch.zeros(3, 224, 224)
    img = Image.open(path).convert("RGB")
    return transform(img)

In [ ]:
#s4
df_temp = df.iloc[0:10].copy().reset_index(drop=True)

#v1

In [ ]:
#s5
SEQ_LEN = 7

for i in range(len(df) - SEQ_LEN):

    #7 IMG window
    window = df_temp.tail(SEQ_LEN).reset_index(drop=True)

    # Build img_seq: shape (1, 7, 3, 224, 224)
    imgs = [load_image(window.loc[j, "img_path"]) for j in range(SEQ_LEN)]
    img_seq = torch.stack(imgs).unsqueeze(0).to(device)  # (1, 7, 3, 224, 224)

    # Build aqi_seq: shape (1, 7, 1)
    aqi_vals = window["AQI"].values.astype(np.float32)      # / 500
    aqi_seq = torch.tensor(aqi_vals).view(1, SEQ_LEN, 1).to(device)

    # Predict
    with torch.no_grad():
        pred = model(img_seq, aqi_seq).squeeze().item() * 310   # * 500


    target_idx = SEQ_LEN + i
    validation_df.at[target_idx, "predicted_aqi"] = round(pred, 2)


    if target_idx + 1 < len(df):
        next_row = df.iloc[target_idx + 1].to_frame().T.reset_index(drop=True)
        df_temp = pd.concat([df_temp.iloc[1:], next_row], ignore_index=True)

#s6
print(validation_df[["Year", "Month", "Date", "AQI", "predicted_aqi"]].iloc[7:20])

#s7
eval_df = validation_df.dropna(subset=["predicted_aqi"]).copy()

actual = eval_df["AQI"].values
predicted = eval_df["predicted_aqi"].values

mae = np.mean(np.abs(actual - predicted))

rmse = np.sqrt(np.mean((actual - predicted) ** 2))

mape = np.mean(np.abs((actual - predicted) / actual)) * 100
ss_res = np.sum((actual - predicted) ** 2)
ss_tot = np.sum((actual - np.mean(actual)) ** 2)
r2 = 1 - (ss_res / ss_tot)

max_err = np.max(np.abs(actual - predicted))

print(f"MAE       : {mae:.2f}  (avg error in AQI units)")
print(f"RMSE      : {rmse:.2f}  (large error sensitive)")
print(f"MAPE      : {mape:.2f}% (avg % error)")
print(f"R²        : {r2:.4f}  (1.0 = perfect fit)")
print(f"Max Error : {max_err:.2f}  (worst prediction)")

    Year Month  Date  AQI  predicted_aqi
7   2025   NOV     8  155         138.42
8   2025   NOV     9  162         130.99
9   2025   NOV    10  158         130.79
10  2025   NOV    11  152         131.25
11  2025   NOV    12  147         126.01
12  2025   NOV    13  150         120.65
13  2025   NOV    14  170         122.08
14  2025   NOV    15  165         125.71
15  2025   NOV    16  168         140.02
16  2025   NOV    17  166         134.36
17  2025   NOV    18  200         128.22
18  2025   NOV    19  162         131.20
19  2025   NOV    20  162         132.33
MAE       : 36.62  (avg error in AQI units)
RMSE      : 54.46  (large error sensitive)
MAPE      : 26.42% (avg % error)
R²        : -0.1938  (1.0 = perfect fit)
Max Error : 214.92  (worst prediction)
